# Controlled Transformer capacity comparison

Compare width 16 / feedforward 32 with width 32 / feedforward 64.
Keep one layer, two heads, context ten, dropout, folds, seed, and training rules fixed.
Use the completed width-16 Transformer as the reference. Do not rerun it.
This changes the capacity recipe. It does not test deeper models or longer context.
Checkpoint selection uses development validation folds. The final test remains reserved.
Read docs/transformer-capacity.md before enabling preparation or training.


In [ ]:
import json
import logging
import os
import shutil
import sys
from pathlib import Path

from IPython import get_ipython
from IPython.display import display

logging.basicConfig(level=logging.INFO, stream=sys.stdout, force=True)
logger = logging.getLogger("transformer_capacity_notebook")
ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "configs/transformer-capacity-grouped-validation.yaml").exists()),
    Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", Path.cwd() / "sleep-classifier")),
).resolve()
ROOT = Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", str(ROOT))).expanduser().resolve()
REPOSITORY_URL = "https://github.com/rbl-bidilatimotei/sleep-classifier.git"
BRANCH = "experiment/transformer-capacity"  # Select the PR branch before merge
RUN_SETUP = False
RUN_CHECK = True
RUN_PREPARATION = False
RUN_TRAINING = False
logger.info("Repository: %s", ROOT)
logger.info("Run setup: %s", RUN_SETUP)
logger.info("Run preparation: %s", RUN_PREPARATION)
logger.info("Run training: %s", RUN_TRAINING)

## Optional setup

Use the current checkout locally. Select experiment/transformer-capacity while reviewing this PR.
Use main after merge. Existing checkouts are not reset or switched automatically.
On a fresh machine, prepare the source cohort features through notebook 07 first.


In [ ]:
if RUN_SETUP:
    if not ROOT.exists():
        !git clone --branch {BRANCH} {REPOSITORY_URL} "{ROOT}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Repository clone failed")
    if shutil.which("uv") is None:
        installer = ROOT / "artifacts/uv-install.sh"
        installer.parent.mkdir(parents=True, exist_ok=True)
        !curl -LsSf https://astral.sh/uv/install.sh -o "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv download failed")
        !sh "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv installation failed")
        os.environ["PATH"] = str(Path.home() / ".local/bin") + os.pathsep + os.environ["PATH"]
    %cd "{ROOT}"
    !uv sync --locked --group gru --group notebook --group dev
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Environment setup failed")


## Preflight without training

Load development features and run a no-gradient forward pass.
The random model has no reported classification performance. No checkpoint or optimizer is created.


In [ ]:
if RUN_CHECK:
    %cd "{ROOT}"
    !uv run --locked --group gru --group notebook python -m sleep_classifier.neural_training \
        --config configs/transformer-width-32.yaml --check
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Transformer preflight failed")

## Inspect the shared fold plan

Planning reads source metadata only. It does not train or prepare feature observations.
The same split seed and participant pool reproduce the measured width-16 fold membership.


In [ ]:
%cd "{ROOT}"
!uv run --locked --group gru --group notebook python -m sleep_classifier.grouped_validation \
    --config configs/transformer-capacity-grouped-validation.yaml --plan
if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("Transformer fold planning failed")

## Prepare candidate fold tables

Enable RUN_PREPARATION and execute the next cell once for a fresh output directory.
This copies unscaled development features and freezes configurations and hashes. It does not fit a model.
If this checkout already has a prepared candidate manifest, leave preparation disabled.


In [ ]:
if RUN_PREPARATION:
    !uv run --locked --group gru --group notebook python \
        -m sleep_classifier.grouped_validation \
        --config configs/transformer-capacity-grouped-validation.yaml --prepare
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Transformer fold preparation failed")
else:
    logger.info("Preparation skipped")

## Train the five candidate models when ready

Enable RUN_TRAINING, execute the settings cell, then execute this training cell interactively.
At most five fits and 100 epochs are requested. Each fold restores its best validation checkpoint.
Partial results are preserved after failure. Resume is not supported.


In [ ]:
if RUN_TRAINING:
    !uv run --locked --group gru --group notebook python -u \
        -m sleep_classifier.grouped_validation \
        --config configs/transformer-capacity-grouped-validation.yaml --train
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Transformer grouped training failed; inspect the partial summary")
else:
    logger.info("Training skipped: enable RUN_TRAINING and execute this cell when ready")

## Inspect experiment status

Missing candidate artifacts are explicitly marked not_started.
A completed width-16 reference does not mean width-32 training has run.


In [ ]:
import pandas as pd
import yaml

from sleep_classifier.paired_architecture_report import PairedArchitectureReport

config_path = ROOT / "configs/transformer-capacity-grouped-validation.yaml"
settings = yaml.safe_load(config_path.read_text())
reference_path = ROOT / "artifacts/transformer-grouped-validation/summary.json"
candidate_path = (config_path.parent / settings["output_dir"] / "summary.json").resolve()
statuses = []
for model, path in [("Width 16", reference_path), ("Width 32", candidate_path)]:
    saved = json.loads(path.read_text()) if path.exists() else {}
    aggregate = saved.get("aggregate", {})
    statuses.append({"model": model, "status": saved.get("status", "not_started"),
                     "completed_folds": len(saved.get("results", [])),
                     "mean_fold_macro_f1": aggregate.get("mean_fold_macro_f1"),
                     "pooled_macro_f1": aggregate.get("pooled_macro_f1"),
                     "pooled_accuracy": aggregate.get("pooled_accuracy")})
display(pd.DataFrame(statuses).set_index("model"))
comparison = None
if all(row["status"] == "complete" for row in statuses):
    comparison = PairedArchitectureReport(reference_path, candidate_path).build()
    output = ROOT / "artifacts/transformer-capacity-comparison/summary.json"
    output.parent.mkdir(parents=True, exist_ok=True)
    output.write_text(json.dumps(comparison, indent=2) + "\n")
    display(pd.DataFrame(comparison["folds"]).set_index("fold"))
    display(pd.DataFrame([comparison["paired"]]))
    for model, key in [("Width 16", "reference"), ("Width 32", "candidate")]:
        logger.info("%s pooled development class scores", model)
        display(pd.DataFrame(comparison[key]["classes"]).T)
    people = [{"participant_id": row["participant_id"], "fold": row["fold"],
               "reference_macro_f1": row["reference"]["macro_f1"],
               "candidate_macro_f1": row["candidate"]["macro_f1"],
               "macro_f1_difference": row["macro_f1_difference"]}
              for row in comparison["participants"]]
    display(pd.DataFrame(people).set_index("participant_id"))
    logger.info("Validated paired comparison saved: %s", output)
else:
    logger.info("A paired score comparison requires both grouped experiments to be complete")

In [ ]:
if comparison is not None:
    import matplotlib.pyplot as plt
    import numpy as np

    figure, axes = plt.subplots(2, 2, figsize=(12, 8))
    pairs = pd.DataFrame(comparison["folds"]).set_index("fold")
    pairs[["reference_macro_f1", "candidate_macro_f1"]].plot.bar(ax=axes[0, 0], ylim=(0, 1))
    axes[0, 0].set(xlabel="Development fold", ylabel="Selected-checkpoint macro-F1")
    class_scores = pd.DataFrame({
        "Width 16": {key: value["f1"] for key, value in comparison["reference"]["classes"].items()},
        "Width 32": {key: value["f1"]
                        for key, value in comparison["candidate"]["classes"].items()},
    })
    class_scores.index = ["Wake", "Light", "Deep", "REM"]
    class_scores.plot.bar(ax=axes[0, 1], ylim=(0, 1))
    axes[0, 1].set(xlabel="Stage", ylabel="Pooled development F1")
    for axis, name, key in [(axes[1, 0], "Width 16", "reference"),
                            (axes[1, 1], "Width 32", "candidate")]:
        counts = np.asarray(comparison[key]["confusion_matrix"])
        fractions = counts / np.maximum(counts.sum(axis=1, keepdims=True), 1)
        axis.imshow(fractions, vmin=0, vmax=1, cmap="Blues")
        axis.set_xticks(range(4), ["Wake", "Light", "Deep", "REM"], rotation=20)
        axis.set_yticks(range(4), ["Wake", "Light", "Deep", "REM"])
        axis.set(title=name, xlabel="Predicted stage", ylabel="True stage")
        for row in range(4):
            for column in range(4):
                axis.text(column, row, f"{fractions[row, column]:.0%}",
                          ha="center", va="center",
                          color="white" if fractions[row, column] > 0.5 else "black")
    figure.tight_layout()
    plt.show()

## Export HTML

The report runs preflight and reads saved results. It never launches training.


In [ ]:
!uv run --locked --group gru --group notebook python scripts/execute_notebook.py \
    --notebook notebooks/13_transformer_capacity.ipynb
if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("Transformer report export failed")